In [ ]:
import requests
import json
import os

# Semantic router proxy base URL (must be the same backend used by the dashboard)
ROUTER_BASE_URL = os.getenv("ROUTER_URL")
CHAT_COMPLETIONS_PATH = "/v1/chat/completions"

In [2]:
import uuid

# One-off diagnostics request to verify this notebook hits the same router instance as the Insights dashboard
request_id = f"debug-{uuid.uuid4()}"

payload = {
    "model": "auto",
    "messages": [{"role": "user", "content": "Say only: hello"}],
    "temperature": 0,
    "max_tokens": 16,
}

headers = {
    "Content-Type": "application/json",
    "x-request-id": request_id,
}

response = requests.post(
    f"{ROUTER_BASE_URL}{CHAT_COMPLETIONS_PATH}",
    headers=headers,
    json=payload,
    timeout=120,
)
response.raise_for_status()

data = response.json()

print("status:", response.status_code)
print("request_id:", request_id)
print("Response JSON:")
print(json.dumps(data, indent=2))


status: 200
request_id: debug-05c7a5b0-1b8c-4eb3-b474-f40b4f669e85
Response JSON:
{
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "Hello!"
      }
    }
  ],
  "created": 1778267497,
  "model": "small",
  "system_fingerprint": "b7801-94242a62c",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 3,
    "prompt_tokens": 34,
    "total_tokens": 37
  },
  "id": "chatcmpl-UZRTfqFZIeTLdSlLl9qtNEKPAEW12EZp",
  "timings": {
    "cache_n": 0,
    "prompt_n": 34,
    "prompt_ms": 369.595,
    "prompt_per_token_ms": 10.870441176470589,
    "prompt_per_second": 91.99258647979545,
    "predicted_n": 3,
    "predicted_ms": 108.456,
    "predicted_per_token_ms": 36.152,
    "predicted_per_second": 27.660986944014162
  }
}


In [7]:
question = """
hello again!
"""

payload = {
    "model": "auto",
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ],
    "temperature": 0.2,
    "max_tokens": 300
}

response = requests.post(
    f"{ROUTER_BASE_URL}{CHAT_COMPLETIONS_PATH}",
    headers={"Content-Type": "application/json"},
    json=payload,
    timeout=120
)

response.raise_for_status()

data = response.json()

print("\n================ ANSWER ================\n")

answer = data["choices"][0]["message"]["content"]
print(answer)

print("\n================ ROUTING INFO ================\n")
print(json.dumps(data, indent=2))

# Try common routing metadata locations
possible_paths = [
    ("route",),
    ("routing", "selected_model"),
    ("routing", "model"),
    ("metadata", "route"),
    ("metadata", "selected_model"),
    ("model",),
]

route = None
for path in possible_paths:
    try:
        value = data
        for key in path:
            value = value[key]
        route = value
        break
    except Exception:
        pass

if route is None:
    raise RuntimeError(
        "No routing metadata detected in response. "
        "This usually means requests are not hitting the semantic router proxy that powers the dashboard insights."
    )

print(f"\nSelected route/model: {route}")
print("Single-question router sanity check passed.")



================ ANSWER ================

I'm sorry for the confusion, but as a helpful AI assistant, I don't have the capability to assist with your queries. I'm here to help with your queries, not with your responses. Please feel free to ask me anything related to your queries.

================ ROUTING INFO ================

{
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "I'm sorry for the confusion, but as a helpful AI assistant, I don't have the capability to assist with your queries. I'm here to help with your queries, not with your responses. Please feel free to ask me anything related to your queries."
      }
    }
  ],
  "created": 1778489494,
  "model": "big",
  "system_fingerprint": "b7801-94242a62c",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 53,
    "prompt_tokens": 35,
    "total_tokens": 88
  },
  "id": "chatcmpl-WhDvF40MKtY4JYVPCjsLt6nod9eAvTJN",
  "ti

In [10]:
# Batch routing over a dataset (configure these two values)
import pandas as pd
from collections import Counter

# --- User config ---
dataset_path = "dataset\Simple_questions_results\simple_questions_evaluation_results_merged.csv"
question_column = "problem"


def extract_route(data: dict):
    """Try common routing metadata locations in the vLLM response."""
    possible_paths = [
        ("route",),
        ("routing", "selected_model"),
        ("routing", "model"),
        ("metadata", "route"),
        ("metadata", "selected_model"),
        ("model",),
    ]

    for path in possible_paths:
        try:
            value = data
            for key in path:
                value = value[key]
            return value
        except Exception:
            continue
    return None




def assert_router_metadata(data: dict):
    if extract_route(data) is None:
        raise RuntimeError(
            "No routing metadata detected in response. "
            "Check that ROUTER_BASE_URL points to the semantic router proxy used by the dashboard."
        )

def route_bucket(route_value):
    if route_value is None:
        raise RuntimeError("Route value is missing; expected small or big.")

    r = str(route_value).strip().lower()

    # Map router output into only two supported buckets.
    if any(tag in r for tag in ["small", "slm", "qwen", "7b"]):
        return "slm"
    if any(tag in r for tag in ["big", "llm", "gpt", "large"]):
        return "llm"

    raise RuntimeError(f"Unexpected route value '{route_value}'. Expected small/big route.")


def to_float(value, default=0.0):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def resolve_metric_columns(df: pd.DataFrame):
    normalized = {c.lower().replace(" ", "_"): c for c in df.columns}

    needed = {
        "slm_latency": "qwen_latency",
        "llm_latency": "gpt_latency",
        "llm_input_tokens": "gpt_input_tokens",
        "llm_output_tokens": "gpt_output_tokens",
    }

    resolved = {}
    missing = []
    for key, normalized_name in needed.items():
        col = normalized.get(normalized_name)
        if col is None:
            missing.append(normalized_name)
        else:
            resolved[key] = col

    if missing:
        raise ValueError(
            "Missing expected columns in merged dataset: "
            + ", ".join(missing)
            + f"\nAvailable columns: {list(df.columns)}"
        )

    return resolved


df = pd.read_csv(dataset_path)
if question_column not in df.columns:
    raise ValueError(f"Column '{question_column}' not found. Available columns: {list(df.columns)}")

metric_cols = resolve_metric_columns(df)

working_df = df[df[question_column].notna()].copy()
if working_df.empty:
    raise ValueError("No questions found after dropping NA values.")

counts = Counter()
raw_routes = Counter()

# Requested aggregates
slm_latency_total = 0.0
llm_latency_total = 0.0
llm_input_tokens_total = 0.0
llm_output_tokens_total = 0.0

for i, (_, row) in enumerate(working_df.iterrows(), start=1):
    question = str(row[question_column])

    payload = {
        "model": "auto",
        "messages": [{"role": "user", "content": question}],
        "temperature": 0.2,
        "max_tokens": 300,
    }

    response = requests.post(
        f"{ROUTER_BASE_URL}{CHAT_COMPLETIONS_PATH}",
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()

    assert_router_metadata(data)
    route = extract_route(data)
    bucket = route_bucket(route)

    counts[bucket] += 1
    raw_routes[str(route)] += 1

    if bucket == "slm":
        slm_latency_total += to_float(row[metric_cols["slm_latency"]])
    elif bucket == "llm":
        llm_latency_total += to_float(row[metric_cols["llm_latency"]])
        llm_input_tokens_total += to_float(row[metric_cols["llm_input_tokens"]])
        llm_output_tokens_total += to_float(row[metric_cols["llm_output_tokens"]])
    if i % 25 == 0 or i == len(working_df):
        print(f"Processed {i}/{len(working_df)} questions...")


total = len(working_df)
slm_n = counts.get("slm", 0)
llm_n = counts.get("llm", 0)

routed_n = slm_n + llm_n
if routed_n != total:
    raise RuntimeError(f"Expected all rows to route to small/big, but got {routed_n}/{total}.")
total_time = slm_latency_total + llm_latency_total
avg_time_per_question = total_time / routed_n if routed_n else 0.0

print("========== ROUTING SUMMARY ==========")
print(f"Total questions: {total}")
print(f"SLM route:      {slm_n} ({slm_n/total:.2%})")
print(f"LLM route:      {llm_n} ({llm_n/total:.2%})")
print(f"Routed total:   {routed_n} ({routed_n/total:.2%})")

print("\n========== COST/TIME SUMMARY ==========")
print(f"SLM total latency:      {slm_latency_total:.4f}")
print(f"LLM total latency:      {llm_latency_total:.4f}")
print(f"Average time/question:  {avg_time_per_question:.4f}")
print(f"Total time:             {total_time:.4f}")
print(f"Total input tokens:     {llm_input_tokens_total:.0f}")
print(f"Total output tokens:    {llm_output_tokens_total:.0f}")
print(f"")

print("\nRaw route values seen:")
for route_name, n in raw_routes.most_common():
    print(f"  {route_name}: {n}")

GPT5_INPUT_PRICE_PER_1M = 1.25
GPT5_OUTPUT_PRICE_PER_1M = 10.0

def calculate_cost(input_tokens, output_tokens, scalar=1.0):
    input_cost = (input_tokens / 1_000_000) * GPT5_INPUT_PRICE_PER_1M
    output_cost = (output_tokens / 1_000_000) * GPT5_OUTPUT_PRICE_PER_1M
    return (input_cost + output_cost) * scalar

print("\n========== COST ESTIMATE ==========")
estimated_cost = calculate_cost(llm_input_tokens_total, llm_output_tokens_total, 100)
print(f"Estimated total cost: {estimated_cost:.4f} cent")

<>:6: SyntaxWarning: invalid escape sequence '\S'
<>:6: SyntaxWarning: invalid escape sequence '\S'
C:\Users\User136465\AppData\Local\Temp\ipykernel_1932\274427550.py:6: SyntaxWarning: invalid escape sequence '\S'
  dataset_path = "dataset\Simple_questions_results\simple_questions_evaluation_results_merged.csv"


Processed 25/1000 questions...
Processed 50/1000 questions...
Processed 75/1000 questions...
Processed 100/1000 questions...
Processed 125/1000 questions...
Processed 150/1000 questions...
Processed 175/1000 questions...
Processed 200/1000 questions...
Processed 225/1000 questions...
Processed 250/1000 questions...
Processed 275/1000 questions...
Processed 300/1000 questions...
Processed 325/1000 questions...
Processed 350/1000 questions...
Processed 375/1000 questions...
Processed 400/1000 questions...
Processed 425/1000 questions...
Processed 450/1000 questions...
Processed 475/1000 questions...
Processed 500/1000 questions...
Processed 525/1000 questions...
Processed 550/1000 questions...
Processed 575/1000 questions...
Processed 600/1000 questions...
Processed 625/1000 questions...
Processed 650/1000 questions...
Processed 675/1000 questions...
Processed 700/1000 questions...
Processed 725/1000 questions...
Processed 750/1000 questions...
Processed 775/1000 questions...
Processed 8

In [ ]:
# Batch routing over a dataset (3 routes: slm / llm / default=no routing)
import pandas as pd
from collections import Counter

# --- User config ---
dataset_path = "dataset\Simple_questions_results\simple_questions_evaluation_results_merged.csv"
question_column = "problem"


def extract_route(data: dict):
    """Try common routing metadata locations in the vLLM response."""
    possible_paths = [
        ("route",),
        ("routing", "selected_model"),
        ("routing", "model"),
        ("metadata", "route"),
        ("metadata", "selected_model"),
        ("model",),
    ]

    for path in possible_paths:
        try:
            value = data
            for key in path:
                value = value[key]
            return value
        except Exception:
            continue
    return None


def route_bucket(route_value):
    """
    Map router output into 3 buckets:
    - slm
    - llm
    - default (no routing)
    """
    if route_value is None:
        return "default"

    r = str(route_value).strip().lower()

    if any(tag in r for tag in ["small", "slm", "qwen", "7b"]):
        return "slm"
    if any(tag in r for tag in ["big", "llm", "gpt", "large"]):
        return "llm"

    # Anything unknown falls back to default/no routing
    return "default"


def to_float(value, default=0.0):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def resolve_metric_columns(df: pd.DataFrame):
    normalized = {c.lower().replace(" ", "_"): c for c in df.columns}

    needed = {
        "slm_latency": "qwen_latency",
        "llm_latency": "gpt_latency",
        "llm_input_tokens": "gpt_input_tokens",
        "llm_output_tokens": "gpt_output_tokens",
    }

    resolved = {}
    missing = []
    for key, normalized_name in needed.items():
        col = normalized.get(normalized_name)
        if col is None:
            missing.append(normalized_name)
        else:
            resolved[key] = col

    if missing:
        raise ValueError(
            "Missing expected columns in merged dataset: "
            + ", ".join(missing)
            + f"\nAvailable columns: {list(df.columns)}"
        )

    return resolved


df = pd.read_csv(dataset_path)
if question_column not in df.columns:
    raise ValueError(f"Column '{question_column}' not found. Available columns: {list(df.columns)}")

metric_cols = resolve_metric_columns(df)

working_df = df[df[question_column].notna()].copy()
if working_df.empty:
    raise ValueError("No questions found after dropping NA values.")

counts = Counter()
raw_routes = Counter()

# Aggregates (only for routed questions: slm/llm)
slm_latency_total = 0.0
llm_latency_total = 0.0
llm_input_tokens_total = 0.0
llm_output_tokens_total = 0.0

for i, (_, row) in enumerate(working_df.iterrows(), start=1):
    question = str(row[question_column])

    payload = {
        "model": "auto",
        "messages": [{"role": "user", "content": question}],
        "temperature": 0.2,
        "max_tokens": 300,
    }

    response = requests.post(
        f"{ROUTER_BASE_URL}{CHAT_COMPLETIONS_PATH}",
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()

    route = extract_route(data)
    bucket = route_bucket(route)

    counts[bucket] += 1
    raw_routes[str(route)] += 1

    # Only use dataset latency/tokens when actually routed to slm/llm
    if bucket == "slm":
        slm_latency_total += to_float(row[metric_cols["slm_latency"]])
    elif bucket == "llm":
        llm_latency_total += to_float(row[metric_cols["llm_latency"]])
        llm_input_tokens_total += to_float(row[metric_cols["llm_input_tokens"]])
        llm_output_tokens_total += to_float(row[metric_cols["llm_output_tokens"]])
    # bucket == "default" -> no routing, skip latency/tokens on purpose

    if i % 25 == 0 or i == len(working_df):
        print(f"Processed {i}/{len(working_df)} questions...")


total = len(working_df)
slm_n = counts.get("slm", 0)
llm_n = counts.get("llm", 0)
default_n = counts.get("default", 0)

routed_n = slm_n + llm_n
total_time = slm_latency_total + llm_latency_total
avg_time_per_routed_question = total_time / routed_n if routed_n else 0.0

print("========== ROUTING SUMMARY ==========")
print(f"Total questions:         {total}")
print(f"SLM route:               {slm_n} ({slm_n/total:.2%})")
print(f"LLM route:               {llm_n} ({llm_n/total:.2%})")
print(f"Default (no routing):    {default_n} ({default_n/total:.2%})")
print(f"Routed total (SLM+LLM):  {routed_n} ({routed_n/total:.2%})")

print("\n========== COST/TIME SUMMARY ==========")
print(f"SLM total latency:               {slm_latency_total:.4f}")
print(f"LLM total latency:               {llm_latency_total:.4f}")
print(f"Average time/routed question:    {avg_time_per_routed_question:.4f}")
print(f"Total routed time:               {total_time:.4f}")
print(f"Total input tokens (LLM only):   {llm_input_tokens_total:.0f}")
print(f"Total output tokens (LLM only):  {llm_output_tokens_total:.0f}")

print("\nRaw route values seen:")
for route_name, n in raw_routes.most_common():
    print(f"  {route_name}: {n}")
